### Hyperparameter Tuning and Training for Generalized Additive Models (GAM)

This notebook performs hyperparameter tuning for GAMs using pygam package.
No interaction terms are used to ensure fair comparison with EBM and NAM.


In [270]:
import numpy as np
import pandas as pd
import json
import time
import sys
import os
import pickle
from pathlib import Path
from sklearn.metrics import mean_squared_error, roc_auc_score
from pygam import LinearGAM, LogisticGAM, s, f
import warnings
warnings.filterwarnings('ignore')

# Add src to path for imports
project_root = Path.cwd().parent.parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import neural_additive_models.data_utils as data_utils
from utils import create_fold_indices
from hp_tuning_utils import sample_hyperparameters
from neural_additive_models.data_utils import get_train_val_test_split


## Configuration


In [271]:
# Dataset configuration
OPENML_DATASET_ID = 31
TASK_TYPE = "classification"  # Options: "regression" or "classification"
dataset_name = f'OpenML_{OPENML_DATASET_ID}_{TASK_TYPE}'
is_regression = (TASK_TYPE == "regression")

# Hyperparameter search space for GAM
hp_search_space = {
    'n_splines': [6, 8, 12, 16, 20, 24],
    'lam': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0],
    'spline_order': [2, 3, 4, 5],
}

# Fixed hyperparameters
fixed_hp = {
    'max_iter': 100,
    'tol': 1e-4,
}

# Tuning parameters
n_trials = 50
random_seed = 42

# Set results directory
results_dir = project_root / 'results' / 'hyperparameter_tuning' / 'gam'
results_dir.mkdir(parents=True, exist_ok=True)


## Load Dataset and Create Folds


In [272]:
# Load dataset
print(f"Loading dataset: {dataset_name}")
data_x, data_y, column_names = data_utils.load_dataset(dataset_name)

# Determine if regression from dataset name
if '_regression' in dataset_name:
    is_regression = True
elif '_classification' in dataset_name:
    is_regression = False

print(f"Dataset shape: {data_x.shape}")
print(f"Target shape: {data_y.shape}")
print(f"Number of features: {data_x.shape[1]}")
print(f"Task type: {'Regression' if is_regression else 'Classification'}")

# Split into train/val/test (matching NAM approach: 60% train, 20% val, 20% test)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = get_train_val_test_split(
    data_x, data_y,
    test_size=0.2,
    val_size=0.2,
    stratified=not is_regression,
    random_state=random_seed
)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")


Loading dataset: OpenML_31_classification
Dataset shape: (1000, 61)
Target shape: (1000,)
Number of features: 61
Task type: Classification
Train: 600, Val: 200, Test: 200


## Hyperparameter Sampling


In [273]:
# Generate hyperparameter configurations
hyperparameters = []
for trial in range(n_trials):
    trial_seed = random_seed + trial
    hp_config = sample_hyperparameters(hp_search_space, trial_seed)
    hyperparameters.append({
        'trial': trial + 1,
        'hyperparameters': hp_config
    })

print(f"Generated {n_trials} hyperparameter configurations")


Generated 50 hyperparameter configurations


## Hyperparameter Tuning


In [ ]:
def train_and_evaluate_gam(X_train, y_train, X_val, y_val, hyperparameters, is_regression=True):
    """Train GAM and return validation score and convergence status."""
    
    n_features = X_train.shape[1]
    n_splines = hyperparameters['n_splines']
    lam = hyperparameters['lam']
    spline_order = hyperparameters['spline_order']
    
    # Create GAM terms - one smooth term per feature (no interactions)
    term_list = [s(i, n_splines=n_splines, spline_order=spline_order) for i in range(n_features)]
    if len(term_list) == 1:
        terms = term_list[0]
    else:
        terms = term_list[0]
        for term in term_list[1:]:
            terms = terms + term
    
    # Create and train GAM
    if is_regression:
        gam = LinearGAM(terms=terms, lam=lam, fit_intercept=True, **fixed_hp)
    else:
        gam = LogisticGAM(terms=terms, lam=lam, fit_intercept=True, **fixed_hp)
    
    start_time = time.time()

    gam.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Check convergence status
    converged = getattr(gam, 'converged_', True)
    
    # Predict and evaluate
    if is_regression:
        y_pred = gam.predict(X_val)
        score = np.sqrt(mean_squared_error(y_val, y_pred))  # RMSE for regression
    else:
        y_pred_proba = gam.predict_proba(X_val)
        score = roc_auc_score(y_val, y_pred_proba)  # AUC for classification
    
    return score, training_time, gam, converged

# Perform hyperparameter tuning
print("=" * 70)
print(f"HYPERPARAMETER TUNING - {n_trials} trials")
print("=" * 70)

trial_results = []

for trial_data in hyperparameters:
    trial_num = trial_data['trial']
    hp = trial_data['hyperparameters']
    
    print(f"\nTrial {trial_num}/{n_trials}...", end=' ', flush=True)
    
    try:
        score, training_time, model, converged = train_and_evaluate_gam(
            X_train, y_train, X_val, y_val, hp, is_regression=is_regression
        )
        
        metric_name = 'RMSE' if is_regression else 'AUC'
        if not converged:
            print(f"did not converge")
        print(f"{metric_name}: {score:.4f} ({training_time:.1f}s)")
        
        trial_results.append({
            'trial': trial_num,
            'hyperparameters': hp,
            'validation_score': score,
            'training_time': training_time,
            'converged': converged,
            'success': True
        })
    except Exception as e:
        print(f"Failed: {str(e)[:50]}")
        trial_results.append({
            'trial': trial_num,
            'hyperparameters': hp,
            'validation_score': None,
            'training_time': None,
            'success': False,
            'error': str(e)
        })

print("\n" + "=" * 70)
print("TUNING COMPLETE")
print("=" * 70)


HYPERPARAMETER TUNING - 50 trials

Trial 1/50... 

In [ ]:
df_results = pd.DataFrame(trial_results)

# Filter successful trials
df_success = df_results[df_results['success']].copy()

if len(df_success) > 0:
    # Filter out non-converging trials when selecting best hyperparameters
    # Non-converging models may be unstable even if they have good scores
    df_converged = df_success[df_success['converged']].copy() if 'converged' in df_success.columns else df_success.copy()
    
    if len(df_converged) > 0:
        # Find best hyperparameters from converged trials only
        if is_regression:
            # Lower is better for RMSE
            best_idx = df_converged['validation_score'].idxmin()
        else:
            # Higher is better for AUC
            best_idx = df_converged['validation_score'].idxmax()
        
        best_trial = df_converged.loc[best_idx]
        used_converged_only = True
    else:
        # Fallback: if no trials converged, use the best from all trials
        print("WARNING: No trials converged! Using best from all trials (may be unstable).")
        if is_regression:
            best_idx = df_success['validation_score'].idxmin()
        else:
            best_idx = df_success['validation_score'].idxmax()
        best_trial = df_success.loc[best_idx]
        used_converged_only = False
    
    print(f"Best trial: {int(best_trial['trial'])}")
    if 'converged' in best_trial and not best_trial['converged']:
        print("WARNING: Best trial did not converge! Model may be unstable.")
    metric_name = 'RMSE' if is_regression else 'AUC'
    print(f"Best {metric_name}: {best_trial['validation_score']:.4f}")
    print(f"Training time: {best_trial['training_time']:.1f}s")
    if 'converged' in best_trial:
        print(f"Converged: {best_trial['converged']}")
    print("\nBest hyperparameters:")
    for key, value in best_trial['hyperparameters'].items():
        print(f"  {key}: {value}")
    
    # Extract best hyperparameters
    best_hp = best_trial['hyperparameters']
    
    # Save best hyperparameters
    best_hp_file = results_dir / f"best_hp_{dataset_name.replace('/', '_').replace(':', '_')}.json"
    with open(best_hp_file, 'w') as f:
        json.dump(best_hp, f, indent=2)
    print(f"\nSaved best hyperparameters to: {best_hp_file}")
    
    # Save all results
    results_file = results_dir / f"tuning_results_{dataset_name.replace('/', '_').replace(':', '_')}.json"
    with open(results_file, 'w') as f:
        json.dump(trial_results, f, indent=2)
    print(f"Saved all results to: {results_file}")
    
    # Display summary statistics
    print("\n" + "=" * 70)
    print("SUMMARY STATISTICS")
    print("=" * 70)
    print(f"Successful trials: {len(df_success)}/{n_trials}")
    if 'converged' in df_success.columns:
        n_converged = df_success['converged'].sum()
        print(f"Converged trials: {n_converged}/{len(df_success)}")
        if n_converged < len(df_success):
            print(f"Non-converged trials: {len(df_success) - n_converged} (excluded from best HP selection)")
    print(f"Mean {metric_name}: {df_success['validation_score'].mean():.4f}")
    print(f"Std {metric_name}: {df_success['validation_score'].std():.4f}")
    print(f"Mean training time: {df_success['training_time'].mean():.1f}s")
else:
    print("No successful trials!")


Best trial: 19
Best AUC: 0.9952
Training time: 0.7s
Converged: True

Best hyperparameters:
  n_splines: 12
  lam: 1.0
  spline_order: 3

Saved best hyperparameters to: c:\Users\dejvi\Documents\pythonProject\neural-additive-models-xai-seminar-1\results\hyperparameter_tuning\gam\best_hp_OpenML_15_classification.json
Saved all results to: c:\Users\dejvi\Documents\pythonProject\neural-additive-models-xai-seminar-1\results\hyperparameter_tuning\gam\tuning_results_OpenML_15_classification.json

SUMMARY STATISTICS
Successful trials: 50/50
Converged trials: 50/50
Mean AUC: 0.9924
Std AUC: 0.0014
Mean training time: 1.0s


## Cross-Validation Evaluation


In [ ]:
# Define cross-validation parameters
NUM_FOLDS = 5

# Create fold indices for cross-validation (same as EBM and NAM)
fold_train_indices, fold_test_indices = create_fold_indices(
    data_x, data_y, num_folds=NUM_FOLDS, random_state=42
)

print(f"Created {NUM_FOLDS} folds for cross-validation")
print(f"Note: GAMs don't use early stopping, so we train one model per fold")


Created 5 folds for cross-validation
Note: GAMs don't use early stopping, so we train one model per fold


## Train Models Across All Folds (5 folds)

This section trains GAM models across all 5 folds. Unlike NAM/EBM which use 20 splits per fold for early stopping, GAMs train one model per fold since they don't require early stopping.


In [ ]:
def gather_gam_predictions(
    fold_train_indices,
    fold_test_indices,
    data_x,
    data_y,
    column_names,
    best_hp,
    fixed_hp,
    is_regression=True,
    num_folds=5,
    base_logdir=None,
):
    """
    Gather GAM predictions across folds.
    
    Unlike NAM/EBM, GAMs don't use early stopping, so we train one model per fold
    on the full training fold data (no need for multiple splits).
    
    Args:
        fold_train_indices: List of train indices for each fold
        fold_test_indices: List of test indices for each fold
        data_x: Full dataset features
        data_y: Full dataset targets
        column_names: List of feature names
        best_hp: Best hyperparameters dict
        fixed_hp: Fixed hyperparameters dict
        is_regression: Whether this is a regression task
        num_folds: Number of folds
        base_logdir: Base directory for saving/loading models (optional)
    
    Returns:
        all_preds_per_fold: List of lists of predictions [fold][split][samples]
        Note: For GAMs, there's only 1 "split" per fold, so [fold][0][samples]
    """
    all_preds_per_fold = [[] for _ in range(num_folds)]
    
    n_features = data_x.shape[1]
    n_splines = best_hp['n_splines']
    lam = best_hp['lam']
    spline_order = best_hp['spline_order']
    
    # Build terms once (same for all models)
    term_list = [s(i, n_splines=n_splines, spline_order=spline_order) for i in range(n_features)]
    if len(term_list) == 1:
        terms = term_list[0]
    else:
        terms = term_list[0]
        for term in term_list[1:]:
            terms = terms + term
    
    for fold in range(1, num_folds + 1):
        fold_idx = fold - 1
        train_indices = fold_train_indices[fold_idx]
        test_indices = fold_test_indices[fold_idx]
        
        # Get training data for this fold (test set is held out)
        X_train_fold = data_x[train_indices]
        y_train_fold = data_y[train_indices]
        X_test_fold = data_x[test_indices]
        y_test_fold = data_y[test_indices]
        
        print(f"Processing fold {fold} (train: {len(train_indices)}, test: {len(test_indices)})...", end=' ', flush=True)
        
        # Check if model exists
        model_path = None
        if base_logdir:
            model_path = os.path.join(base_logdir, f'fold_{fold}', 'gam_model.pkl')
            os.makedirs(os.path.dirname(model_path), exist_ok=True)
        
        # Load or train model
        if model_path and os.path.exists(model_path):
            with open(model_path, 'rb') as f:
                gam = pickle.load(f)
            print("(loaded)", flush=True)
        else:
            # Train new model on full training fold data (no splits needed for GAMs)
            if is_regression:
                gam = LinearGAM(terms=terms, lam=lam, fit_intercept=True, **fixed_hp)
            else:
                gam = LogisticGAM(terms=terms, lam=lam, fit_intercept=True, **fixed_hp)
            gam.fit(X_train_fold, y_train_fold)
            
            # Save model if path provided
            if model_path:
                with open(model_path, 'wb') as f:
                    pickle.dump(gam, f)
            
            print("(trained)", flush=True)
        
        # Get predictions on test set
        if is_regression:
            preds_test = gam.predict(X_test_fold)
        else:
            # For classification, get probabilities
            preds_test = gam.predict_proba(X_test_fold)
        
        # Store predictions (wrapped in a list to match expected format [fold][split][samples])
        all_preds_per_fold[fold_idx].append(preds_test)
    
    return all_preds_per_fold



In [ ]:
# Load best hyperparameters from tuning results
best_hp_file = results_dir / f'best_hp_{dataset_name.replace("/", "_").replace(":", "_")}.json'

with open(best_hp_file, 'r') as f:
    best_hp = json.load(f)
print(f"Loaded best hyperparameters from: {best_hp_file}")
print(f"Best hyperparameters:")
for key, value in best_hp.items():
    print(f"  {key}: {value}")

# Set results directory for saving models
base_logdir = project_root / 'results' / 'training' / 'gam' / f'openml_{OPENML_DATASET_ID}_{TASK_TYPE}'

print("\n" + "="*70)
print("TRAINING MODELS PER FOLD")
print("="*70)
print(f"Number of folds: {NUM_FOLDS}")
print(f"Total models to train: {NUM_FOLDS}")
print(f"Note: GAMs don't use early stopping, so we train one model per fold")
print(f"      (unlike NAM/EBM which use 20 splits per fold for early stopping)")
print(f"Models will be saved to: {base_logdir}")
print("="*70)
print("\nNote: Models are trained on-demand if they don't already exist.")
print("\nGathering predictions (training models as needed)...")
print("="*70)

all_preds_per_fold = gather_gam_predictions(
    fold_train_indices=fold_train_indices,
    fold_test_indices=fold_test_indices,
    data_x=data_x,
    data_y=data_y,
    column_names=column_names,
    best_hp=best_hp,
    fixed_hp=fixed_hp,
    is_regression=is_regression,
    num_folds=NUM_FOLDS,
    base_logdir=str(base_logdir)
)


Loaded best hyperparameters from: c:\Users\dejvi\Documents\pythonProject\neural-additive-models-xai-seminar-1\results\hyperparameter_tuning\gam\best_hp_OpenML_15_classification.json
Best hyperparameters:
  n_splines: 12
  lam: 1.0
  spline_order: 3

TRAINING MODELS PER FOLD
Number of folds: 5
Total models to train: 5
Note: GAMs don't use early stopping, so we train one model per fold
      (unlike NAM/EBM which use 20 splits per fold for early stopping)
Models will be saved to: c:\Users\dejvi\Documents\pythonProject\neural-additive-models-xai-seminar-1\results\training\gam\openml_15_classification

Note: Models are trained on-demand if they don't already exist.

Gathering predictions (training models as needed)...
Processing fold 1 (train: 559, test: 140)... (trained)
Processing fold 2 (train: 559, test: 140)... (trained)
Processing fold 3 (train: 559, test: 140)... (trained)
Processing fold 4 (train: 559, test: 140)... (trained)
Processing fold 5 (train: 560, test: 139)... (trained)


## Save Performance Metrics

Save detailed performance metrics in the same format as EBM/NAM for fair comparison.


In [ ]:
from utils import save_performance_metrics

# Save detailed performance metrics for statistical analysis
output_file = save_performance_metrics(
    all_preds_per_fold=all_preds_per_fold,
    fold_test_indices=fold_test_indices,
    data_y=data_y,
    dataset_id=OPENML_DATASET_ID,
    dataset_name=dataset_name,
    task_type=TASK_TYPE,
    model_type='GAM',
    num_folds=NUM_FOLDS,
    num_splits=1,  # GAMs don't use early stopping, so only 1 model per fold
    is_regression=is_regression,
    project_root=project_root,
    verbose=True
)



PERFORMANCE METRICS SAVED
Saved to: c:\Users\dejvi\Documents\pythonProject\neural-additive-models-xai-seminar-1\results\evaluation\gam_OpenML_15_classification_performance.json
Summary:
  Mean AUC: 0.9923
  Std AUC: 0.0038
  Min AUC: 0.9869
  Max AUC: 0.9971
